# Prepare KNN (NMSLIB) for stackoverflow_all: brute-force and NAPP

(skip for now)

Run this notebook before `05_stackoverflow_all_tuning_bm25_and_model1.ipynb`.

This notebook will:
1. Ensure the unified collection `stackoverflow_all` prerequisites exist.
2. Ensure a BM25 extractor JSON exists for sparse vector export.
3. Export sparse vectors for NMSLIB.
4. Create `cand_prov.json` files for both `bruteforce` and `napp` provider modes.
5. Print query-server commands for brute-force and NAPP startup.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
COLLECT_NAME = 'stackoverflow_all'

BM25_QUERY_FIELD = 'text'
BM25_INDEX_FIELD = 'text'
BM25_K1 = 1.2 # use from best finetuned bm25
BM25_B = 0.8

NMSLIB_URI = 'localhost:9090'
QUERY_SERVER_PORT = 9090

EXPORT_REL_PATH = 'nmslib/bm25_text_text/export.data'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

collect_dir = COLLECT_ROOT / COLLECT_NAME
exper_desc_dir = collect_dir / 'exper_desc'
exper_desc_best_dir = collect_dir / 'exper_desc.best'

extractor_rel_path = f'exper_desc/bm25tune_{BM25_QUERY_FIELD}_{BM25_INDEX_FIELD}/bm25tune_k1={BM25_K1}_b={BM25_B}.json'
extractor_abs_path = collect_dir / extractor_rel_path

nmslib_bruteforce_conf_rel = 'exper_desc.best/nmslib/bruteforce/cand_prov.json'
nmslib_napp_conf_rel = 'exper_desc.best/nmslib/napp/cand_prov.json'

nmslib_bruteforce_conf_abs = collect_dir / nmslib_bruteforce_conf_rel
nmslib_napp_conf_abs = collect_dir / nmslib_napp_conf_rel

print('REPO_ROOT            =', REPO_ROOT)
print('SCRIPTS_DIR          =', SCRIPTS_DIR)
print('COLLECT_ROOT         =', COLLECT_ROOT)
print('COLLECT_NAME         =', COLLECT_NAME)
print('COLLECT_DIR          =', collect_dir)
print('Extractor (rel)      =', extractor_rel_path)
print('Extractor (abs)      =', extractor_abs_path)
print('Bruteforce conf (rel)=', nmslib_bruteforce_conf_rel)
print('NAPP conf (rel)      =', nmslib_napp_conf_rel)
print('NMSLIB_URI           =', NMSLIB_URI)
print('EXPORT_REL_PATH      =', EXPORT_REL_PATH)

REPO_ROOT            = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR          = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT         = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
COLLECT_NAME         = stackoverflow_all
COLLECT_DIR          = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all
Extractor (rel)      = exper_desc/bm25tune_text_text/bm25tune_k1=1.2_b=0.8.json
Extractor (abs)      = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc/bm25tune_text_text/bm25tune_k1=1.2_b=0.8.json
Bruteforce conf (rel)= exper_desc.best/nmslib/bruteforce/cand_prov.json
NAPP conf (rel)      = exper_desc.best/nmslib/napp/cand_prov.json
NMSLIB_URI           = localhost:9090
EXPORT_REL_PATH      = nmslib/bm25_text_text/

In [2]:
required_paths = [
    collect_dir,
    collect_dir / 'input_data' / 'train' / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / 'train' / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / 'dev1' / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / 'dev1' / 'AnswerFields.jsonl',
    collect_dir / 'lucene_index',
    collect_dir / 'forward_index' / BM25_INDEX_FIELD
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

print('All prerequisites for NMSLIB preparation are present.')

All prerequisites for NMSLIB preparation are present.


## Ensure BM25 extractor JSON exists
If the extractor file is missing, this cell generates BM25 tuning descriptors and extractors.

In [ ]:
exper_desc_dir.mkdir(parents=True, exist_ok=True)

if not extractor_abs_path.exists():
    cmd = [
        'python3', './gen_exper_desc/gen_bm25_tune_json_desc.py',
        '--index_field_name', BM25_INDEX_FIELD,
        '--query_field_name', BM25_QUERY_FIELD,
        '--outdir', str(exper_desc_dir),
        '--exper_subdir', 'tuning',
        '--rel_desc_path', 'exper_desc'
    ]
    print('Generating BM25 descriptors/extractors:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'BM25 descriptor generation failed: {res.returncode}')

if not extractor_abs_path.exists():
    raise FileNotFoundError(f'Expected extractor file was not created: {extractor_abs_path}')

print('Extractor is ready:', extractor_abs_path)

## Export sparse vectors for NMSLIB

In [ ]:
(collect_dir / 'derived_data' / 'nmslib' / 'bm25_text_text').mkdir(parents=True, exist_ok=True)

cmd = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    extractor_rel_path,
    EXPORT_REL_PATH
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'NMSLIB sparse export failed: {res.returncode}')

export_abs = collect_dir / 'derived_data' / EXPORT_REL_PATH
if not export_abs.exists():
    raise FileNotFoundError(f'Expected export file was not created: {export_abs}')

print('Export file ready:', export_abs)

## Create candidate-provider configs for notebook 05
Both configs intentionally point to the same extractor; server mode (`brute_force` vs `napp`) is selected when starting the NMSLIB query server.

In [ ]:
nmslib_bruteforce_conf_abs.parent.mkdir(parents=True, exist_ok=True)
nmslib_napp_conf_abs.parent.mkdir(parents=True, exist_ok=True)

cand_prov_payload = {
    'extrType': extractor_rel_path,
    'sparseInterleave': 'false'
}

for conf_path in [nmslib_bruteforce_conf_abs, nmslib_napp_conf_abs]:
    with conf_path.open('w', encoding='utf-8') as f:
        json.dump(cand_prov_payload, f, indent=2)
    print('Wrote:', conf_path)

print('Bruteforce conf ready:', nmslib_bruteforce_conf_abs)
print('NAPP conf ready      :', nmslib_napp_conf_abs)

## Start query server (run in a separate terminal)
Use one mode at a time, then select corresponding `PROVIDER_MODE` in notebook 05.

In [ ]:
export_abs = collect_dir / 'derived_data' / EXPORT_REL_PATH

cmd_bruteforce = f'''export COLLECT_ROOT={COLLECT_ROOT}
<path_to_query_server>/query_server \
  -p {QUERY_SERVER_PORT} \
  -s negdotprod_sparse_bin_fast \
  -i {export_abs} \
  -m brute_force'''

cmd_napp = f'''export COLLECT_ROOT={COLLECT_ROOT}
<path_to_query_server>/query_server \
  -p {QUERY_SERVER_PORT} \
  -s negdotprod_sparse_bin_fast \
  -i {export_abs} \
  -m napp'''

print('----- Brute-force server command -----')
print(cmd_bruteforce)
print()
print('----- NAPP server command -----')
print(cmd_napp)
print()
print('For notebook 05 setup:')
print("- use PROVIDER_MODE='nmslib_bruteforce' for brute-force")
print("- use PROVIDER_MODE='nmslib_napp' for NAPP")
print(f"- keep NMSLIB_URI='{NMSLIB_URI}'")
print(f"- brute-force conf: {nmslib_bruteforce_conf_rel}")
print(f"- napp conf      : {nmslib_napp_conf_rel}")